In [3]:
!pip install lightgbm 

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 969.1 kB/s eta 0:00:01
   --------------- ------------------------ 0.5/1.4 MB 969.1 kB/s eta 0:00:01
   -------------------------------------- - 1.3/1.4 MB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.4 MB/s  0:00:01



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# ==============================================================================
# Weather & Flood Risk Forecasting Model — 48h-ahead multi-target training script
# (v6 — Calibrated probability thresholding for extreme event detection)
# ==============================================================================

import json
import os
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, median_absolute_error, r2_score,
    accuracy_score, f1_score, classification_report, confusion_matrix
)

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    GBM_BACKEND = "lightgbm"
except ImportError:
    from sklearn.ensemble import (
        GradientBoostingRegressor as LGBMRegressor,
        GradientBoostingClassifier as LGBMClassifier,
    )
    GBM_BACKEND = "sklearn_gbm (lightgbm not installed — falling back)"

warnings.filterwarnings("ignore")
print(f"Gradient boosting backend in use: {GBM_BACKEND}")

# ------------------------------------------------------------------------
# 1. Config
# ------------------------------------------------------------------------
DATA_PATH = "weather_flood_dataset.csv"
OUTPUT_DIR = "model_artifacts"
RANDOM_STATE = 42

INTERVAL_MIN = 15
HORIZON_STEPS = int(48 * 60 / INTERVAL_MIN)   # 192 steps = 48h ahead
BLOCK_STEPS = int(7 * 24 * 60 / INTERVAL_MIN)  # 672 steps = 1 week per block
EMBARGO_STEPS = HORIZON_STEPS                  # purge one horizon at each split edge

# Calibrated decision threshold for minority 'heavy' rain class
HEAVY_RAIN_THRESHOLD = 0.2

TIME_FEATURES = ["hour_sin", "hour_cos", "doy_sin", "doy_cos", "month"]
RAW_SENSOR_FEATURES = [
    "temperature", "humidity", "temp_bmp180", "pressure_loc1",
    "temp_bmp280", "pressure_loc2", "rain_raw", "soil_moisture",
]
ENGINEERED_FEATURES = [
    "pressure_loc1_trend_3h", "pressure_loc1_trend_6h",
    "pressure_loc2_trend_3h", "pressure_loc2_trend_6h", "pressure_diff",
    "humidity_trend_3h", "rain_rate", "rain_accum_6h", "rain_accum_24h",
]
FEATURE_COLUMNS = TIME_FEATURES + RAW_SENSOR_FEATURES + ENGINEERED_FEATURES

DIRECT_REGRESSION_TARGETS = ["temperature_target", "humidity_target"]

PRESSURE_PAIRS = [
    ("pressure_loc1", "pressure_loc1_target", "pressure_loc1_target_delta"),
    ("pressure_loc2", "pressure_loc2_target", "pressure_loc2_target_delta"),
]
DELTA_REGRESSION_TARGETS = [d for (_, _, d) in PRESSURE_PAIRS]

REGRESSION_TARGETS = DIRECT_REGRESSION_TARGETS + DELTA_REGRESSION_TARGETS
CLASSIFICATION_TARGETS = ["rain_category_target", "flood_risk_target"]

RAW_TARGET_COLUMNS = [
    "temperature_target", "humidity_target",
    "pressure_loc1_target", "pressure_loc2_target",
    "rain_category_target", "flood_risk_target",
]

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------------------
# 2. Load Data
# ------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

missing_cols = [c for c in FEATURE_COLUMNS + RAW_TARGET_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing expected columns: {missing_cols}")

n_missing = df[FEATURE_COLUMNS + RAW_TARGET_COLUMNS].isna().sum().sum()
if n_missing > 0:
    print(f"Found {n_missing} missing values across features/targets — forward/back-filling.")
    df[FEATURE_COLUMNS + RAW_TARGET_COLUMNS] = (
        df[FEATURE_COLUMNS + RAW_TARGET_COLUMNS].ffill().bfill()
    )

print(f"Dataset shape: {df.shape}")

# ------------------------------------------------------------------------
# 3. Build Delta Pressure Targets + Outlier Cleaning
# ------------------------------------------------------------------------
print("\n--- Building delta pressure targets & cleaning outliers ---")
for now_col, target_col, delta_col in PRESSURE_PAIRS:
    df[delta_col] = df[target_col] - df[now_col]

    q1, q3 = df[delta_col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 5 * iqr, q3 + 5 * iqr
    outlier_mask = (df[delta_col] < lo) | (df[delta_col] > hi)
    num_outliers = int(outlier_mask.sum())
    if num_outliers > 0:
        print(f"  {delta_col}: replacing {num_outliers} outliers "
              f"(outside [{lo:.1f}, {hi:.1f}] Pa) via linear interpolation.")
        df.loc[outlier_mask, delta_col] = np.nan
        df[delta_col] = df[delta_col].interpolate(method="linear").ffill().bfill()

# ------------------------------------------------------------------------
# 4. Blocked, Stratified Split with Embargo Purging
# ------------------------------------------------------------------------
print("\n--- Building blocked, stratified train/val/test split ---")
df["block_id"] = np.arange(len(df)) // BLOCK_STEPS

block_dominant_label = (
    df.groupby("block_id")["flood_risk_target"]
    .agg(lambda s: s.mode().iat[0] if not s.mode().empty else "unknown")
)
block_ids = block_dominant_label.index.values
block_labels = block_dominant_label.values

train_blocks, rest_blocks, _, rest_labels = train_test_split(
    block_ids, block_labels,
    test_size=(1 - TRAIN_FRAC),
    stratify=block_labels,
    random_state=RANDOM_STATE,
)
val_blocks, test_blocks = train_test_split(
    rest_blocks,
    test_size=(1 - TRAIN_FRAC - VAL_FRAC) / (1 - TRAIN_FRAC),
    stratify=rest_labels,
    random_state=RANDOM_STATE,
)

block_to_split = {}
block_to_split.update({b: "train" for b in train_blocks})
block_to_split.update({b: "val" for b in val_blocks})
block_to_split.update({b: "test" for b in test_blocks})
df["split"] = df["block_id"].map(block_to_split)

split_arr = df["split"].values
n = len(df)
change_points = np.where(split_arr[1:] != split_arr[:-1])[0] + 1
embargo_mask = np.zeros(n, dtype=bool)
for cp in change_points:
    lo = max(0, cp - EMBARGO_STEPS)
    hi = min(n, cp + EMBARGO_STEPS)
    embargo_mask[lo:hi] = True

n_embargoed = int(embargo_mask.sum())
print(f"  Dropping {n_embargoed} rows ({n_embargoed / n:.1%}) inside the "
      f"{EMBARGO_STEPS}-step embargo window around split boundaries.")

df_clean = df.loc[~embargo_mask].copy()

train_df = df_clean[df_clean["split"] == "train"].copy()
val_df = df_clean[df_clean["split"] == "val"].copy()
test_df = df_clean[df_clean["split"] == "test"].copy()

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

X_train, X_val, X_test = (
    train_df[FEATURE_COLUMNS], val_df[FEATURE_COLUMNS], test_df[FEATURE_COLUMNS]
)

# ------------------------------------------------------------------------
# 5. Scale Features
# ------------------------------------------------------------------------
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=FEATURE_COLUMNS, index=X_train.index
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val), columns=FEATURE_COLUMNS, index=X_val.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=FEATURE_COLUMNS, index=X_test.index
)

joblib.dump(scaler, f"{OUTPUT_DIR}/feature_scaler.joblib")

# ------------------------------------------------------------------------
# 6. Train Regression Models
# ------------------------------------------------------------------------
regression_models = {}
regression_metrics = {}

reg_params = dict(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    random_state=RANDOM_STATE,
)

print("\n===== Training regression models =====")
for target in REGRESSION_TARGETS:
    y_train = train_df[target]
    y_val = val_df[target]
    y_test = test_df[target]

    if GBM_BACKEND == "lightgbm":
        import lightgbm as lgb
        model = LGBMRegressor(
            **reg_params, colsample_bytree=0.8, verbosity=-1,
            objective="huber",
        )
        model.fit(
            X_train_scaled, y_train,
            eval_set=[(X_val_scaled, y_val)],
            eval_metric="mae",
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )
    else:
        model = LGBMRegressor(
            **reg_params, loss="huber", validation_fraction=0.15,
            n_iter_no_change=30, tol=1e-4,
        )
        model.fit(X_train_scaled, y_train)

    preds_test = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, preds_test)
    medae = median_absolute_error(y_test, preds_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds_test))
    r2 = r2_score(y_test, preds_test)

    regression_models[target] = model
    regression_metrics[target] = {"MAE": mae, "MedAE": medae, "RMSE": rmse, "R2": r2}
    print(
        f"  {target:30s} -> MAE={mae:.3f}  MedAE={medae:.3f}  "
        f"RMSE={rmse:.3f}  R2={r2:.3f}"
    )

    joblib.dump(model, f"{OUTPUT_DIR}/model_{target}.joblib")

print("\n  Reconstructed absolute pressure error (current + predicted delta):")
for now_col, target_col, delta_col in PRESSURE_PAIRS:
    preds_delta = regression_models[delta_col].predict(X_test_scaled)
    reconstructed = test_df[now_col].values + preds_delta
    mae_abs = mean_absolute_error(test_df[target_col], reconstructed)
    print(f"    {target_col:26s} -> reconstructed MAE={mae_abs:.3f} Pa")

# ------------------------------------------------------------------------
# 7. Train Classification Models (with Calibrated Thresholding)
# ------------------------------------------------------------------------
classification_models = {}
classification_encoders = {}
classification_metrics = {}

print("\n===== Training classification models =====")
for target in CLASSIFICATION_TARGETS:
    encoder = LabelEncoder()
    encoder.fit(df[target].dropna())

    y_train_enc = encoder.transform(train_df[target])
    y_val_enc = encoder.transform(val_df[target])
    y_test_enc = encoder.transform(test_df[target])

    all_label_ids = np.arange(len(encoder.classes_))

    if GBM_BACKEND == "lightgbm":
        import lightgbm as lgb
        
        # Class weighting balanced + custom tree depth settings
        model = LGBMClassifier(
            n_estimators=800,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            verbosity=-1,
            class_weight="balanced",
            min_child_samples=5,
            random_state=RANDOM_STATE
        )
        model.fit(
            X_train_scaled, y_train_enc,
            eval_set=[(X_val_scaled, y_val_enc)],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )
    else:
        model = LGBMClassifier(n_estimators=500, learning_rate=0.03, max_depth=4, random_state=RANDOM_STATE)
        model.fit(X_train_scaled, y_train_enc)

    # Predict raw class probabilities
    probs_test = model.predict_proba(X_test_scaled)

    # Apply optimal decision threshold for rare heavy rain events
    if target == "rain_category_target" and "heavy" in encoder.classes_:
        heavy_idx = np.where(encoder.classes_ == "heavy")[0][0]
        preds_test_enc = np.argmax(probs_test, axis=1)
        
        # Override argmax with calibrated threshold (0.35)
        heavy_mask = probs_test[:, heavy_idx] >= HEAVY_RAIN_THRESHOLD
        preds_test_enc[heavy_mask] = heavy_idx
    else:
        preds_test_enc = np.argmax(probs_test, axis=1)

    acc = accuracy_score(y_test_enc, preds_test_enc)
    f1_macro = f1_score(y_test_enc, preds_test_enc, average="macro", zero_division=0)
    report = classification_report(
        y_test_enc, preds_test_enc,
        labels=all_label_ids, target_names=encoder.classes_, zero_division=0,
    )
    cm = confusion_matrix(y_test_enc, preds_test_enc, labels=all_label_ids)

    classification_models[target] = model
    classification_encoders[target] = encoder
    classification_metrics[target] = {"accuracy": acc, "f1_macro": f1_macro}

    print(f"\n  {target} -> accuracy={acc:.3f}  f1_macro={f1_macro:.3f}")
    print(f"  Classes: {list(encoder.classes_)}")
    print("  Classification report:")
    print(report)
    print("  Confusion matrix (rows=true, cols=pred):")
    print(cm)

    joblib.dump(model, f"{OUTPUT_DIR}/model_{target}.joblib")
    joblib.dump(encoder, f"{OUTPUT_DIR}/encoder_{target}.joblib")

# ------------------------------------------------------------------------
# 8. Export Manifest & Prediction Helper
# ------------------------------------------------------------------------
manifest = {
    "feature_columns": FEATURE_COLUMNS,
    "regression_targets": REGRESSION_TARGETS,
    "pressure_pairs": PRESSURE_PAIRS,
    "classification_targets": CLASSIFICATION_TARGETS,
    "block_steps": BLOCK_STEPS,
    "embargo_steps": EMBARGO_STEPS,
    "heavy_rain_threshold": HEAVY_RAIN_THRESHOLD,
    "gbm_backend": GBM_BACKEND,
    "regression_metrics": regression_metrics,
    "classification_metrics": classification_metrics,
}
with open(f"{OUTPUT_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"\nAll models, scaler, encoders, and manifest saved under ./{OUTPUT_DIR}/")


def predict_all_targets(new_data: pd.DataFrame) -> pd.DataFrame:
    """
    Predicts all targets 48 hours ahead with calibrated probability thresholds 
    for severe weather events.
    """
    X_new = new_data[FEATURE_COLUMNS]
    X_new_scaled = pd.DataFrame(
        scaler.transform(X_new), columns=FEATURE_COLUMNS, index=X_new.index
    )

    results = {}

    # Direct regressions (temperature, humidity)
    for target in DIRECT_REGRESSION_TARGETS:
        results[target] = regression_models[target].predict(X_new_scaled)

    # Reconstructed absolute pressure from predicted deltas
    for now_col, target_col, delta_col in PRESSURE_PAIRS:
        pred_delta = regression_models[delta_col].predict(X_new_scaled)
        results[target_col] = new_data[now_col].values + pred_delta

    # Classifications with calibrated probability thresholding
    for target, model in classification_models.items():
        encoder = classification_encoders[target]
        probs = model.predict_proba(X_new_scaled)
        
        preds_enc = np.argmax(probs, axis=1)

        # Apply calibrated threshold for heavy rain in production inference
        if target == "rain_category_target" and "heavy" in encoder.classes_:
            heavy_idx = np.where(encoder.classes_ == "heavy")[0][0]
            heavy_mask = probs[:, heavy_idx] >= HEAVY_RAIN_THRESHOLD
            preds_enc[heavy_mask] = heavy_idx

        results[target] = encoder.inverse_transform(preds_enc)

    return pd.DataFrame(results, index=new_data.index)

Gradient boosting backend in use: lightgbm
Dataset shape: (20000, 29)

--- Building delta pressure targets & cleaning outliers ---
  pressure_loc1_target_delta: replacing 49 outliers (outside [-3060.0, 3056.0] Pa) via linear interpolation.
  pressure_loc2_target_delta: replacing 45 outliers (outside [-3069.0, 3069.0] Pa) via linear interpolation.

--- Building blocked, stratified train/val/test split ---
  Dropping 6528 rows (32.6%) inside the 192-step embargo window around split boundaries.
Train: 10208 | Val: 1152 | Test: 2112

===== Training regression models =====
  temperature_target             -> MAE=0.549  MedAE=0.445  RMSE=0.691  R2=0.970
  humidity_target                -> MAE=2.541  MedAE=2.121  RMSE=3.182  R2=0.946
  pressure_loc1_target_delta     -> MAE=289.057  MedAE=250.945  RMSE=359.971  R2=0.005
  pressure_loc2_target_delta     -> MAE=287.716  MedAE=249.055  RMSE=356.532  R2=0.020

  Reconstructed absolute pressure error (current + predicted delta):
    pressure_loc1_t

In [14]:
# ==============================================================================
# Inference Test Script — Weather & Flood Risk 48h-Ahead Forecasting
# Loads trained model artifacts and tests prediction pipeline with synthetic inputs
# ==============================================================================

import json
import joblib
import numpy as np
import pandas as pd

ARTIFACTS_DIR = "model_artifacts"

# ------------------------------------------------------------------------
# 1. Load Saved Manifest & Artifacts
# ------------------------------------------------------------------------
with open(f"{ARTIFACTS_DIR}/manifest.json", "r") as f:
    manifest = json.load(f)

scaler = joblib.load(f"{ARTIFACTS_DIR}/feature_scaler.joblib")

regression_models = {
    target: joblib.load(f"{ARTIFACTS_DIR}/model_{target}.joblib")
    for target in manifest["regression_targets"]
}

classification_models = {
    target: joblib.load(f"{ARTIFACTS_DIR}/model_{target}.joblib")
    for target in manifest["classification_targets"]
}

classification_encoders = {
    target: joblib.load(f"{ARTIFACTS_DIR}/encoder_{target}.joblib")
    for target in manifest["classification_targets"]
}

feature_cols = manifest["feature_columns"]
heavy_threshold = manifest.get("heavy_rain_threshold", 0.20)

print("✅ All artifacts loaded successfully!")
print(f"   Target features: {len(feature_cols)}")
print(f"   Heavy rain decision threshold: {heavy_threshold}\n")

# ------------------------------------------------------------------------
# 2. Generate Synthetic Sensor Inputs (5 Sample Timesteps)
# ------------------------------------------------------------------------
np.random.seed(42)
N_SAMPLES = 5

sample_data = {
    # Time features
    "hour_sin": np.sin(2 * np.pi * np.array([0, 6, 12, 18, 22]) / 24),
    "hour_cos": np.cos(2 * np.pi * np.array([0, 6, 12, 18, 22]) / 24),
    "doy_sin": np.sin(2 * np.pi * np.ones(N_SAMPLES) * 150 / 365),
    "doy_cos": np.cos(2 * np.pi * np.ones(N_SAMPLES) * 150 / 365),
    "month": np.full(N_SAMPLES, 6),
    # Raw sensor measurements
    "temperature": np.random.uniform(18.0, 32.0, N_SAMPLES),
    "humidity": np.random.uniform(55.0, 95.0, N_SAMPLES),
    "temp_bmp180": np.random.uniform(18.5, 31.5, N_SAMPLES),
    "pressure_loc1": np.random.uniform(100500.0, 101800.0, N_SAMPLES),
    "temp_bmp280": np.random.uniform(18.2, 31.8, N_SAMPLES),
    "pressure_loc2": np.random.uniform(100400.0, 101700.0, N_SAMPLES),
    "rain_raw": np.random.choice([0.0, 2.5, 12.0, 45.0], size=N_SAMPLES, p=[0.6, 0.2, 0.1, 0.1]),
    "soil_moisture": np.random.uniform(20.0, 85.0, N_SAMPLES),
    # Engineered trend & accumulation features
    "pressure_loc1_trend_3h": np.random.uniform(-150.0, 150.0, N_SAMPLES),
    "pressure_loc1_trend_6h": np.random.uniform(-300.0, 300.0, N_SAMPLES),
    "pressure_loc2_trend_3h": np.random.uniform(-150.0, 150.0, N_SAMPLES),
    "pressure_loc2_trend_6h": np.random.uniform(-300.0, 300.0, N_SAMPLES),
    "pressure_diff": np.random.uniform(-100.0, 100.0, N_SAMPLES),
    "humidity_trend_3h": np.random.uniform(-10.0, 10.0, N_SAMPLES),
    "rain_rate": np.random.uniform(0.0, 25.0, N_SAMPLES),
    "rain_accum_6h": np.random.uniform(0.0, 50.0, N_SAMPLES),
    "rain_accum_24h": np.random.uniform(0.0, 120.0, N_SAMPLES),
}

df_test_input = pd.DataFrame(sample_data)

# ------------------------------------------------------------------------
# 3. Predict All Targets (Production Inference Function)
# ------------------------------------------------------------------------
def predict_all_targets(new_data: pd.DataFrame) -> pd.DataFrame:
    X_new = new_data[feature_cols]
    X_new_scaled = pd.DataFrame(
        scaler.transform(X_new), columns=feature_cols, index=X_new.index
    )

    results = {}

    # Direct regressions (temperature, humidity)
    for target in manifest["regression_targets"]:
        if target in ["temperature_target", "humidity_target"]:
            results[target] = regression_models[target].predict(X_new_scaled)

    # Reconstructed absolute pressure from predicted deltas
    for now_col, target_col, delta_col in manifest["pressure_pairs"]:
        pred_delta = regression_models[delta_col].predict(X_new_scaled)
        results[target_col] = new_data[now_col].values + pred_delta

    # Classification targets with calibrated thresholding
    for target in manifest["classification_targets"]:
        model = classification_models[target]
        encoder = classification_encoders[target]
        probs = model.predict_proba(X_new_scaled)

        preds_enc = np.argmax(probs, axis=1)

        # Apply calibrated threshold for heavy rain
        if target == "rain_category_target" and "heavy" in encoder.classes_:
            heavy_idx = np.where(encoder.classes_ == "heavy")[0][0]
            heavy_mask = probs[:, heavy_idx] >= heavy_threshold
            preds_enc[heavy_mask] = heavy_idx

        results[target] = encoder.inverse_transform(preds_enc)

    return pd.DataFrame(results, index=new_data.index)


# Run predictions
df_predictions = predict_all_targets(df_test_input)

# ------------------------------------------------------------------------
# 4. Display Formatted Results
# ------------------------------------------------------------------------
print("=" * 80)
print(" 🔮 48-HOUR AHEAD WEATHER & FLOOD FORECAST (TEST RUN)")
print("=" * 80)

for idx in range(N_SAMPLES):
    print(f"\n--- Sample Timestep #{idx + 1} ---")
    print(f" Current Sensor Inputs:")
    print(f"   Temp: {df_test_input.loc[idx, 'temperature']:.1f}°C | "
          f"Humidity: {df_test_input.loc[idx, 'humidity']:.1f}% | "
          f"Pressure (Loc1): {df_test_input.loc[idx, 'pressure_loc1']:.1f} Pa")
    print(f" Predictions (48 Hours Ahead):")
    print(f"   🌡️ Predicted Temp:          {df_predictions.loc[idx, 'temperature_target']:.2f} °C")
    print(f"   💧 Predicted Humidity:      {df_predictions.loc[idx, 'humidity_target']:.2f} %")
    print(f"   📉 Predicted Pressure Loc1:  {df_predictions.loc[idx, 'pressure_loc1_target']:.1f} Pa")
    print(f"   📉 Predicted Pressure Loc2:  {df_predictions.loc[idx, 'pressure_loc2_target']:.1f} Pa")
    print(f"   🌧️ Predicted Rain Category:  {df_predictions.loc[idx, 'rain_category_target'].upper()}")
    print(f"   ⚠️ Predicted Flood Risk:    {df_predictions.loc[idx, 'flood_risk_target'].upper()}")

print("\n" + "=" * 80)
print("Pipeline execution completed without errors!")

✅ All artifacts loaded successfully!
   Target features: 22
   Heavy rain decision threshold: 0.2

 🔮 48-HOUR AHEAD WEATHER & FLOOD FORECAST (TEST RUN)

--- Sample Timestep #1 ---
 Current Sensor Inputs:
   Temp: 23.2°C | Humidity: 61.2% | Pressure (Loc1): 100738.4 Pa
 Predictions (48 Hours Ahead):
   🌡️ Predicted Temp:          24.07 °C
   💧 Predicted Humidity:      73.16 %
   📉 Predicted Pressure Loc1:  100726.1 Pa
   📉 Predicted Pressure Loc2:  101398.7 Pa
   🌧️ Predicted Rain Category:  LIGHT
   ⚠️ Predicted Flood Risk:    HIGH

--- Sample Timestep #2 ---
 Current Sensor Inputs:
   Temp: 31.3°C | Humidity: 57.3% | Pressure (Loc1): 100895.5 Pa
 Predictions (48 Hours Ahead):
   🌡️ Predicted Temp:          29.29 °C
   💧 Predicted Humidity:      63.68 %
   📉 Predicted Pressure Loc1:  100883.3 Pa
   📉 Predicted Pressure Loc2:  100637.6 Pa
   🌧️ Predicted Rain Category:  DRY
   ⚠️ Predicted Flood Risk:    HIGH

--- Sample Timestep #3 ---
 Current Sensor Inputs:
   Temp: 28.2°C | Humidity

In [16]:
import json
import joblib
import numpy as np
import pandas as pd

ARTIFACTS_DIR = "model_artifacts"

# 1. Load Manifest & Artifacts
with open(f"{ARTIFACTS_DIR}/manifest.json", "r") as f:
    manifest = json.load(f)

scaler = joblib.load(f"{ARTIFACTS_DIR}/feature_scaler.joblib")
feature_cols = manifest["feature_columns"]
heavy_threshold = manifest.get("heavy_rain_threshold", 0.20)

regression_models = {
    target: joblib.load(f"{ARTIFACTS_DIR}/model_{target}.joblib")
    for target in manifest["regression_targets"]
}

classification_models = {
    target: joblib.load(f"{ARTIFACTS_DIR}/model_{target}.joblib")
    for target in manifest["classification_targets"]
}

classification_encoders = {
    target: joblib.load(f"{ARTIFACTS_DIR}/encoder_{target}.joblib")
    for target in manifest["classification_targets"]
}

# 2. Strict Production Inference Function
def predict_all_targets_strict(new_data: pd.DataFrame) -> pd.DataFrame:
    # CRITICAL: Reorder columns to strictly match feature_columns order from training
    X_new = new_data[feature_cols].copy()
    
    # Scale features
    X_new_scaled = pd.DataFrame(
        scaler.transform(X_new), columns=feature_cols, index=X_new.index
    )

    results = {}

    # Direct regressions
    for target in manifest["regression_targets"]:
        if target in ["temperature_target", "humidity_target"]:
            results[target] = regression_models[target].predict(X_new_scaled)

    # Pressure deltas reconstruction
    for now_col, target_col, delta_col in manifest["pressure_pairs"]:
        pred_delta = regression_models[delta_col].predict(X_new_scaled)
        results[target_col] = new_data[now_col].values + pred_delta

    # Classifications
    for target in manifest["classification_targets"]:
        model = classification_models[target]
        encoder = classification_encoders[target]
        probs = model.predict_proba(X_new_scaled)

        preds_enc = np.argmax(probs, axis=1)

        # Calibrated threshold rule
        if target == "rain_category_target" and "heavy" in encoder.classes_:
            heavy_idx = np.where(encoder.classes_ == "heavy")[0][0]
            heavy_mask = probs[:, heavy_idx] >= heavy_threshold
            preds_enc[heavy_mask] = heavy_idx

        results[target] = encoder.inverse_transform(preds_enc)

    return pd.DataFrame(results, index=new_data.index)

# 3. Test with Explicit Order
realistic_samples = pd.DataFrame([
    {  # Scenario 1: Clear, dry day
        "hour_sin": 0.0, "hour_cos": 1.0, "doy_sin": 0.5, "doy_cos": 0.8, "month": 6,
        "temperature": 28.0, "humidity": 45.0, "temp_bmp180": 28.0, "pressure_loc1": 101300.0,
        "temp_bmp280": 28.0, "pressure_loc2": 101300.0, "rain_raw": 0.0, "soil_moisture": 25.0,
        "pressure_loc1_trend_3h": 10.0, "pressure_loc1_trend_6h": 20.0,
        "pressure_loc2_trend_3h": 10.0, "pressure_loc2_trend_6h": 20.0,
        "pressure_diff": 0.0, "humidity_trend_3h": -2.0, "rain_rate": 0.0,
        "rain_accum_6h": 0.0, "rain_accum_24h": 0.0
    },
    {  # Scenario 2: Torrential downpour & high soil saturation
        "hour_sin": 0.0, "hour_cos": 1.0, "doy_sin": 0.5, "doy_cos": 0.8, "month": 6,
        "temperature": 21.0, "humidity": 95.0, "temp_bmp180": 21.0, "pressure_loc1": 99800.0,
        "temp_bmp280": 21.0, "pressure_loc2": 99800.0, "rain_raw": 45.0, "soil_moisture": 88.0,
        "pressure_loc1_trend_3h": -250.0, "pressure_loc1_trend_6h": -500.0,
        "pressure_loc2_trend_3h": -250.0, "pressure_loc2_trend_6h": -500.0,
        "pressure_diff": 0.0, "humidity_trend_3h": 15.0, "rain_rate": 35.0,
        "rain_accum_6h": 65.0, "rain_accum_24h": 140.0
    }
])

results = predict_all_targets_strict(realistic_samples)

print("--- Re-tested Scenario Results ---")
print(f"Scenario 1 (Dry Day)   -> Rain: {results.loc[0, 'rain_category_target'].upper():8s} | Flood Risk: {results.loc[0, 'flood_risk_target'].upper()}")
print(f"Scenario 2 (Storm Day) -> Rain: {results.loc[1, 'rain_category_target'].upper():8s} | Flood Risk: {results.loc[1, 'flood_risk_target'].upper()}")

--- Re-tested Scenario Results ---
Scenario 1 (Dry Day)   -> Rain: DRY      | Flood Risk: LOW
Scenario 2 (Storm Day) -> Rain: DRY      | Flood Risk: LOW


In [17]:
# Test 3: Atmospheric Precursor to a Future Storm
precursor_samples = pd.DataFrame([
    {  # Scenario A: Normal Calm Day
        "hour_sin": 0.0, "hour_cos": 1.0, "doy_sin": 0.5, "doy_cos": 0.8, "month": 6,
        "temperature": 25.0, "humidity": 50.0, "temp_bmp180": 25.0, "pressure_loc1": 101300.0,
        "temp_bmp280": 25.0, "pressure_loc2": 101300.0, "rain_raw": 0.0, "soil_moisture": 30.0,
        "pressure_loc1_trend_3h": 0.0, "pressure_loc1_trend_6h": 0.0,
        "pressure_loc2_trend_3h": 0.0, "pressure_loc2_trend_6h": 0.0,
        "pressure_diff": 0.0, "humidity_trend_3h": 0.0, "rain_rate": 0.0,
        "rain_accum_6h": 0.0, "rain_accum_24h": 0.0
    },
    {  # Scenario B: Approaching Severe Weather Front (Rapid Barometric Drop + Surging Humidity)
        "hour_sin": 0.0, "hour_cos": 1.0, "doy_sin": 0.5, "doy_cos": 0.8, "month": 6,
        "temperature": 18.0, "humidity": 92.0, "temp_bmp180": 18.0, "pressure_loc1": 99500.0,
        "temp_bmp280": 18.0, "pressure_loc2": 99500.0, "rain_raw": 5.0, "soil_moisture": 75.0,
        "pressure_loc1_trend_3h": -350.0, "pressure_loc1_trend_6h": -700.0,
        "pressure_loc2_trend_3h": -350.0, "pressure_loc2_trend_6h": -700.0,
        "pressure_diff": -50.0, "humidity_trend_3h": 18.0, "rain_rate": 8.0,
        "rain_accum_6h": 25.0, "rain_accum_24h": 60.0
    }
])

results = predict_all_targets_strict(precursor_samples)

print("--- Precursor Signals Test ---")
print(f"Scenario A (Stable Day)     -> 48h Rain: {results.loc[0, 'rain_category_target'].upper():8s} | 48h Flood Risk: {results.loc[0, 'flood_risk_target'].upper()}")
print(f"Scenario B (Incoming Front) -> 48h Rain: {results.loc[1, 'rain_category_target'].upper():8s} | 48h Flood Risk: {results.loc[1, 'flood_risk_target'].upper()}")

--- Precursor Signals Test ---
Scenario A (Stable Day)     -> 48h Rain: DRY      | 48h Flood Risk: LOW
Scenario B (Incoming Front) -> 48h Rain: DRY      | 48h Flood Risk: LOW


In [18]:
# Check Feature Importances for Rain and Flood Risk Models
for target in ["rain_category_target", "flood_risk_target"]:
    model = classification_models[target]
    importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print(f"\nTop 5 Most Important Features for [{target}]:")
    print(importances.head(5).to_string())


Top 5 Most Important Features for [rain_category_target]:
doy_sin           1523
rain_accum_24h    1325
doy_cos           1240
rain_accum_6h     1027
hour_sin           918

Top 5 Most Important Features for [flood_risk_target]:
doy_sin           1249
doy_cos            729
hour_cos           480
rain_accum_24h     449
pressure_loc1      376


In [19]:
import pandas as pd
import numpy as np

# Load original dataset
df_orig = pd.read_csv("weather_flood_dataset.csv")

# Find actual severe storm rows in the dataset
heavy_rain_samples = df_orig[df_orig["rain_category_target"] == "heavy"].head(3)
high_flood_samples = df_orig[df_orig["flood_risk_target"] == "high"].head(3)

print("=" * 70)
print(" 🧪 TESTING REAL HISTORICAL STORM ROWS FROM DATASET")
print("=" * 70)

# Test Real Heavy Rain Rows
print("\n--- Testing Actual 'heavy' Rain Target Rows ---")
preds_heavy = predict_all_targets_strict(heavy_rain_samples)
for idx, (real_idx, row) in enumerate(heavy_rain_samples.iterrows()):
    true_rain = row["rain_category_target"]
    pred_rain = preds_heavy.iloc[idx]["rain_category_target"]
    true_flood = row["flood_risk_target"]
    pred_flood = preds_heavy.iloc[idx]["flood_risk_target"]
    print(f"Row #{real_idx:5d} -> Actual 48h Rain: {true_rain.upper():7s} | Predicted 48h Rain: {pred_rain.upper():7s}")
    print(f"               Actual 48h Flood: {true_flood.upper():7s} | Predicted 48h Flood: {pred_flood.upper():7s}")

# Test Real High Flood Risk Rows
print("\n--- Testing Actual 'high' Flood Risk Target Rows ---")
preds_flood = predict_all_targets_strict(high_flood_samples)
for idx, (real_idx, row) in enumerate(high_flood_samples.iterrows()):
    true_rain = row["rain_category_target"]
    pred_rain = preds_flood.iloc[idx]["rain_category_target"]
    true_flood = row["flood_risk_target"]
    pred_flood = preds_flood.iloc[idx]["flood_risk_target"]
    print(f"Row #{real_idx:5d} -> Actual 48h Rain: {true_rain.upper():7s} | Predicted 48h Rain: {pred_rain.upper():7s}")
    print(f"               Actual 48h Flood: {true_flood.upper():7s} | Predicted 48h Flood: {pred_flood.upper():7s}")

 🧪 TESTING REAL HISTORICAL STORM ROWS FROM DATASET

--- Testing Actual 'heavy' Rain Target Rows ---
Row #12492 -> Actual 48h Rain: HEAVY   | Predicted 48h Rain: HEAVY  
               Actual 48h Flood: HIGH    | Predicted 48h Flood: HIGH   
Row #12493 -> Actual 48h Rain: HEAVY   | Predicted 48h Rain: HEAVY  
               Actual 48h Flood: HIGH    | Predicted 48h Flood: HIGH   
Row #12494 -> Actual 48h Rain: HEAVY   | Predicted 48h Rain: HEAVY  
               Actual 48h Flood: HIGH    | Predicted 48h Flood: HIGH   

--- Testing Actual 'high' Flood Risk Target Rows ---
Row #10976 -> Actual 48h Rain: MODERATE | Predicted 48h Rain: DRY    
               Actual 48h Flood: HIGH    | Predicted 48h Flood: WATCH  
Row #10977 -> Actual 48h Rain: MODERATE | Predicted 48h Rain: DRY    
               Actual 48h Flood: HIGH    | Predicted 48h Flood: WATCH  
Row #10978 -> Actual 48h Rain: LIGHT   | Predicted 48h Rain: DRY    
               Actual 48h Flood: HIGH    | Predicted 48h Flood: WATCH 